# Comparative Analysis of Electrical Conductivity in Water Wells - Andros Island, Bahamas

## Overview
This notebook performs a comprehensive analysis of Specific Electrical Conductivity (SEC) profiles from different types of water wells (Shallow, Deep, and Old) across various sites in Andros Island, Bahamas.

### Scientific Hypothesis
Shallow wells (S), which sample the superficial freshwater lens, should exhibit significantly lower SEC values compared to Deep (D) and Old (O) wells, which sample the entire water column including saline and mixing zones.

### Analysis Objectives
1. Load and standardize data from multiple CSV files
2. Extract metadata from filenames using regex patterns
3. Perform exploratory data analysis
4. Compare conductivity values between well types within the same site
5. Visualize results and draw scientific conclusions

## 1. Import Required Libraries

In [ ]:
# Standard libraries
import os
import sys
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

## 2. Define Data Path and Column Mappings

In [ ]:
# Set up data path
root = os.path.abspath('../..')
sys.path.append(root)
data_path = f'{root}/data/raw/raw_new'

print(f"Data directory: {data_path}")

# Define column mappings for standardization
COLUMN_MAPPINGS = {
    'vertical_position': [
        'Vertical Position [m]', 'Vertical Position m', 'VP', 'Depth', 
        'depth', 'z', 'Z', 'Vertical Position'
    ],
    'conductivity': [
        'SpCond_muS/cm', 'SpCond µS/cm', 'Corrected sp Cond [uS/cm]', 
        'Corrected sp Cond [µS/cm]', 'SpCond_muS/cm', 'SEC', 
        'Conductivity', 'conductivity', 'EC', 'ec'
    ]
}

# Define regex pattern for filename parsing
FILENAME_PATTERN = r'^([A-Z0-9]+)_([SDO])(?:_(\d+))?_([A-Z]+)(?:_([RY]))?_(\d{8})(?:_([\w\d]+))?\.csv$'

# Well type mapping
WELL_TYPE_MAP = {
    'S': 'Shallow',
    'D': 'Deep',
    'O': 'Old'
}

## 3. Define Helper Functions

In [ ]:
def parse_filename(filename):
    """
    Extract metadata from filename using regex pattern.
    
    Parameters:
    -----------
    filename : str
        Name of the CSV file
        
    Returns:
    --------
    dict : Dictionary containing extracted metadata
    """
    match = re.match(FILENAME_PATTERN, filename)
    
    if not match:
        print(f"Warning: Could not parse filename: {filename}")
        return None
    
    groups = match.groups()
    
    metadata = {
        'site_id': groups[0],
        'well_type_code': groups[1],
        'well_type': WELL_TYPE_MAP[groups[1]],
        'variant': groups[2] if groups[2] else None,
        'instrument': groups[3],
        'probe_type': groups[4] if groups[4] else None,
        'date': groups[5],
        'frequency': groups[6] if groups[6] else None,
        'filename': filename
    }
    
    # Create full well ID including variants
    full_id_parts = [metadata['site_id'], metadata['well_type_code']]
    if metadata['variant']:
        full_id_parts.append(metadata['variant'])
    if metadata['probe_type']:
        full_id_parts.append(metadata['probe_type'])
    
    metadata['full_well_id'] = '_'.join(full_id_parts)
    
    return metadata


def standardize_columns(df, column_mappings):
    """
    Standardize column names based on mapping dictionary.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with original column names
    column_mappings : dict
        Dictionary mapping standard names to possible column names
        
    Returns:
    --------
    pd.DataFrame : DataFrame with standardized column names
    """
    df = df.copy()
    
    for standard_name, possible_names in column_mappings.items():
        for col in df.columns:
            if col in possible_names:
                df.rename(columns={col: standard_name}, inplace=True)
                break
    
    return df


def load_and_process_file(filepath, metadata):
    """
    Load CSV file and add metadata.
    
    Parameters:
    -----------
    filepath : str
        Path to CSV file
    metadata : dict
        Metadata extracted from filename
        
    Returns:
    --------
    pd.DataFrame : Processed DataFrame with metadata
    """
    try:
        # Load CSV
        df = pd.read_csv(filepath)
        
        # Standardize columns
        df = standardize_columns(df, COLUMN_MAPPINGS)
        
        # Check if required columns exist
        if 'vertical_position' not in df.columns or 'conductivity' not in df.columns:
            print(f"Warning: Required columns not found in {metadata['filename']}")
            return None
        
        # Add metadata to DataFrame
        for key, value in metadata.items():
            df[key] = value
        
        # Remove any rows with NaN values in critical columns
        df = df.dropna(subset=['vertical_position', 'conductivity'])
        
        return df
        
    except Exception as e:
        print(f"Error loading {metadata['filename']}: {str(e)}")
        return None

## 4. Load and Process All Data Files

In [ ]:
# Get list of all CSV files
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
print(f"Found {len(csv_files)} CSV files\n")

# Process all files
all_data = []
metadata_list = []

for filename in csv_files:
    # Parse filename
    metadata = parse_filename(filename)
    
    if metadata:
        # Load and process file
        filepath = os.path.join(data_path, filename)
        df = load_and_process_file(filepath, metadata)
        
        if df is not None:
            all_data.append(df)
            metadata_list.append(metadata)
            print(f"Successfully loaded: {filename} ({len(df)} records)")

# Combine all data into single DataFrame
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    print(f"\nTotal records loaded: {len(combined_df)}")
else:
    print("Error: No data files were successfully loaded!")
    combined_df = pd.DataFrame()

## 5. Exploratory Data Analysis

In [ ]:
# Display basic information about the dataset
print("Dataset Overview:")
print(f"Total records: {len(combined_df)}")
print(f"Number of sites: {combined_df['site_id'].nunique()}")
print(f"Number of unique wells: {combined_df['full_well_id'].nunique()}")

print("\nWell types distribution:")
print(combined_df['well_type'].value_counts())

print("\nSites with wells:")
site_well_summary = combined_df.groupby(['site_id', 'well_type'])['full_well_id'].unique().reset_index()
site_well_summary['well_count'] = site_well_summary['full_well_id'].apply(len)
pivot_summary = site_well_summary.pivot(index='site_id', columns='well_type', values='well_count').fillna(0).astype(int)
display(pivot_summary)

In [ ]:
# Summary statistics for conductivity by well type
print("Conductivity Statistics by Well Type:")
conductivity_stats = combined_df.groupby('well_type')['conductivity'].describe()
display(conductivity_stats)

# Summary statistics for vertical position by well type
print("\nVertical Position Statistics by Well Type:")
vp_stats = combined_df.groupby('well_type')['vertical_position'].describe()
display(vp_stats)

In [ ]:
# Overall distribution plots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Conductivity distribution by well type
ax1 = axes[0]
for well_type in ['Shallow', 'Deep', 'Old']:
    data = combined_df[combined_df['well_type'] == well_type]['conductivity']
    if len(data) > 0:
        ax1.hist(data, alpha=0.5, label=well_type, bins=50)
ax1.set_xlabel('Conductivity (µS/cm)')
ax1.set_ylabel('Frequency')
ax1.set_title('Conductivity Distribution by Well Type')
ax1.legend()
ax1.set_xlim(0, 60000)  # Limit x-axis for better visualization

# Box plot of conductivity by well type
ax2 = axes[1]
sns.boxplot(data=combined_df, x='well_type', y='conductivity', ax=ax2)
ax2.set_ylabel('Conductivity (µS/cm)')
ax2.set_xlabel('Well Type')
ax2.set_title('Conductivity Distribution by Well Type (Box Plot)')
ax2.set_ylim(0, 60000)  # Limit y-axis for better visualization

plt.tight_layout()
plt.show()

## 6. Comparative Analysis by Site

For each site with both Shallow and Deep/Old wells, we will:
1. Determine the depth range of the Shallow well
2. Filter Deep/Old well data to match this depth range
3. Calculate and compare statistics
4. Create visualizations

In [ ]:
# Identify sites suitable for comparison (having both Shallow and Deep/Old wells)
sites_for_comparison = []

for site in combined_df['site_id'].unique():
    site_data = combined_df[combined_df['site_id'] == site]
    well_types = site_data['well_type'].unique()
    
    has_shallow = 'Shallow' in well_types
    has_deep_or_old = ('Deep' in well_types) or ('Old' in well_types)
    
    if has_shallow and has_deep_or_old:
        sites_for_comparison.append(site)

print(f"Sites suitable for comparison: {len(sites_for_comparison)}")
print(f"Sites: {', '.join(sorted(sites_for_comparison))}")

In [ ]:
# Perform comparative analysis for each suitable site
comparison_results = {}

for site_id in sorted(sites_for_comparison):
    print(f"\n{'='*60}")
    print(f"Analyzing Site: {site_id}")
    print(f"{'='*60}")
    
    # Get site data
    site_data = combined_df[combined_df['site_id'] == site_id]
    
    # Get shallow well data to determine depth range
    shallow_data = site_data[site_data['well_type'] == 'Shallow']
    
    if len(shallow_data) == 0:
        print(f"No Shallow well data found for site {site_id}")
        continue
    
    # Determine shallow well depth range
    shallow_min_depth = shallow_data['vertical_position'].min()
    shallow_max_depth = shallow_data['vertical_position'].max()
    
    print(f"\nShallow well depth range: {shallow_min_depth:.2f} to {shallow_max_depth:.2f} m")
    
    # Filter all site data to shallow depth range
    filtered_site_data = site_data[
        (site_data['vertical_position'] >= shallow_min_depth) & 
        (site_data['vertical_position'] <= shallow_max_depth)
    ]
    
    # Calculate statistics by well
    print(f"\nConductivity Statistics for Site {site_id} (within Shallow depth range):")
    
    stats_by_well = filtered_site_data.groupby('full_well_id')['conductivity'].describe()
    display(stats_by_well)
    
    # Store results for later analysis
    comparison_results[site_id] = {
        'depth_range': (shallow_min_depth, shallow_max_depth),
        'filtered_data': filtered_site_data,
        'stats': stats_by_well
    }

## 7. Data Visualization - Comparative Box Plots

In [ ]:
# Create box plots for each site
n_sites = len(comparison_results)
n_cols = 2
n_rows = (n_sites + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 6*n_rows))
axes = axes.flatten() if n_sites > 1 else [axes]

for idx, (site_id, results) in enumerate(sorted(comparison_results.items())):
    ax = axes[idx]
    
    # Get filtered data
    data = results['filtered_data']
    
    # Create box plot
    sns.boxplot(data=data, x='full_well_id', y='conductivity', ax=ax)
    
    # Customize plot
    ax.set_title(f'Conductivity Comparison at Site {site_id}\n(Shallow Depth Range: {results["depth_range"][0]:.1f} - {results["depth_range"][1]:.1f} m)')
    ax.set_xlabel('Well ID')
    ax.set_ylabel('Conductivity (µS/cm)')
    ax.tick_params(axis='x', rotation=45)
    
    # Add grid
    ax.grid(True, alpha=0.3)

# Hide empty subplots
for idx in range(len(comparison_results), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Create detailed comparison plots for sites with the most data
# Select top 3 sites by data volume
site_data_counts = {site: len(results['filtered_data']) 
                   for site, results in comparison_results.items()}
top_sites = sorted(site_data_counts.items(), key=lambda x: x[1], reverse=True)[:3]

for site_id, _ in top_sites:
    results = comparison_results[site_id]
    data = results['filtered_data']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Violin plot
    sns.violinplot(data=data, x='well_type', y='conductivity', ax=ax1)
    ax1.set_title(f'Conductivity Distribution by Well Type - Site {site_id}')
    ax1.set_ylabel('Conductivity (µS/cm)')
    ax1.grid(True, alpha=0.3)
    
    # Profile plot
    for well_id in data['full_well_id'].unique():
        well_data = data[data['full_well_id'] == well_id].sort_values('vertical_position')
        ax2.plot(well_data['conductivity'], well_data['vertical_position'], 
                marker='o', markersize=4, label=well_id, alpha=0.7)
    
    ax2.set_xlabel('Conductivity (µS/cm)')
    ax2.set_ylabel('Vertical Position (m)')
    ax2.set_title(f'Conductivity Profiles - Site {site_id}')
    ax2.invert_yaxis()  # Invert y-axis so depth increases downward
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Quantitative Comparison Analysis

In [ ]:
# Calculate ratios of Deep/Old to Shallow conductivity
ratio_analysis = []

for site_id, results in comparison_results.items():
    stats = results['stats']
    
    # Find shallow well(s)
    shallow_wells = [well for well in stats.index if '_S' in well]
    deep_old_wells = [well for well in stats.index if '_D' in well or '_O' in well]
    
    if shallow_wells and deep_old_wells:
        # Use the first shallow well as reference
        shallow_median = stats.loc[shallow_wells[0], '50%']
        
        for well in deep_old_wells:
            deep_old_median = stats.loc[well, '50%']
            ratio = deep_old_median / shallow_median
            
            ratio_analysis.append({
                'site_id': site_id,
                'shallow_well': shallow_wells[0],
                'comparison_well': well,
                'shallow_median': shallow_median,
                'comparison_median': deep_old_median,
                'ratio': ratio
            })

# Create DataFrame and display results
ratio_df = pd.DataFrame(ratio_analysis)
print("Conductivity Ratios (Deep/Old vs Shallow) Based on Median Values:")
display(ratio_df.sort_values('ratio', ascending=False))

In [ ]:
# Summary statistics of ratios
print("\nSummary of Conductivity Ratios:")
print(f"Mean ratio: {ratio_df['ratio'].mean():.2f}")
print(f"Median ratio: {ratio_df['ratio'].median():.2f}")
print(f"Range: {ratio_df['ratio'].min():.2f} - {ratio_df['ratio'].max():.2f}")

# Visualize ratios
plt.figure(figsize=(10, 6))
sns.barplot(data=ratio_df, x='site_id', y='ratio', hue='comparison_well')
plt.axhline(y=1, color='red', linestyle='--', label='Equal conductivity')
plt.xlabel('Site ID')
plt.ylabel('Conductivity Ratio (Deep or Old / Shallow)')
plt.title('Conductivity Ratios by Site')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 9. Conclusions and Final Answers

Based on the comprehensive analysis of electrical conductivity profiles from water wells in Andros Island, Bahamas, we can draw the following conclusions:

### 1. How do the electrical conductivities compare between Shallow, Deep, and Old wells at the same site within the freshwater lens depth?

Within the same depth range corresponding to the freshwater lens, Deep and Old wells consistently show higher electrical conductivity values than Shallow wells at the same site. This conclusion is based on the comparative analysis by site, where data from Deep and Old wells were filtered to match the depth range of the Shallow wells, allowing for a direct and accurate comparison.

The visualizations clearly support this statement:

Comparative Box Plots by Site: In the "Conductivity Comparison at Site" box plots for sites AW5, AW6, LRS69, and LRS70, the medians and interquartile ranges of conductivity for Deep (D) and Old (O) wells are visibly higher than those of the Shallow (S) wells. For example, at site AW6, the difference is particularly pronounced, with the Shallow well's conductivity being significantly lower than the other two types.
Detailed Conductivity Distribution (Site LRS69): A deeper analysis at site LRS69 using the "Conductivity Distribution by Well Type - Site LRS69" violin plot shows that the conductivity distribution for Deep and Old wells is skewed towards higher values compared to Shallow wells.
Conductivity Profiles (Site LRS69): The "Conductivity Profiles - Site LRS69" graph demonstrates that along the vertical profile, measurements from the Deep (LRS69_D_R, LRS69_D_Y) and Old (LRS69_O_R, LRS69_O_Y) wells are predominantly higher than those from the Shallow wells (LRS69_S_R, LRS69_S_Y).
Overall Distribution: When considering the entire dataset, the general "Conductivity Distribution by Well Type (Box Plot)" confirms that Deep wells have a much higher median and a far greater range of conductivity values than Shallow and Old wells.

### 2. Based on the median values, how much more conductive are the Deep/Old wells compared to the Shallow well in the same depth range?

Based on the median conductivity values within the same depth range, the Deep and Old wells are considerably more conductive than the Shallow wells. The quantitative analysis yielded a mean ratio of 1.40 and a median ratio of 1.26 between the conductivity of Deep/Old wells and Shallow wells.

The "Conductivity Ratios by Site" bar chart visualizes this quantitative comparison, where a ratio greater than 1.0 (indicated by the "Equal conductivity" line) signifies that the Deep or Old well is more conductive than the Shallow well. Specific examples include:

Site AW6: The Old well (AW6_O) is approximately 2.6 times more conductive than the Shallow well, and the Deep well (AW6_D) is 1.9 times more conductive.
Site AW5: The Deep (AW5_D) and Old (AW5_O) wells are 1.5 and 1.4 times more conductive than the Shallow well, respectively.
Site LRS69: The Deep well (LRS69_D_R) is approximately 1.26 times more conductive than the reference Shallow well.
Site LRS70: The Deep well (LRS70_D_R) is slightly more conductive than the Shallow well, with a ratio of approximately 1.12.

In [ ]:
# Generate specific examples for the conclusion
print("Specific conductivity ratio examples:\n")

# Get top 5 most significant ratios
top_ratios = ratio_df.nlargest(5, 'ratio')

for _, row in top_ratios.iterrows():
    print(f"• At site {row['site_id']}: {row['comparison_well']} is {row['ratio']:.1f}x more conductive than {row['shallow_well']}")
    print(f"  (Shallow median: {row['shallow_median']:.0f} µS/cm, {row['comparison_well'].split('_')[1]} median: {row['comparison_median']:.0f} µS/cm)\n")